# SmolVLM: CORD item and total extraction on a single A100

Train a **new LoRA adapter from `HuggingFaceTB/SmolVLM-Base`**, using only [CORD v2](https://huggingface.co/datasets/naver-clova-ix/cord-v2) and [CORD v1](https://huggingface.co/datasets/naver-clova-ix/cord-v1). No SROIE data or previous adapter is loaded. Run on an A100 40 GB or 80 GB with internet access and enough disk space for both datasets and the base model.

The target is `items` plus the printed final `total`. Each item has description, quantity, unit price, printed line amount, and nested `sub_items` when annotated. Categories are a later task; this notebook does not invent category labels. It learns image → JSON directly, without a separate OCR model.

**Versions overlap:** the authors describe v2 as corrected labels and improved submenu hierarchy. We load both versions, collapse identical RGB images, and prefer v2 labels. Overlapping receipts contribute once, so this is not 2,000 independent receipts. Any duplicate found across splits goes to the more protected split: test > validation > train. Exact image hashing does not detect every near-duplicate; audit the manifest before making generalization claims. [CORD authors' documentation](https://github.com/clovaai/cord)

Start with `MAX_STEPS = 5` for a smoke test. For full training, use `None`. The default BF16 LoRA settings are a starting point, not a measured A100 memory guarantee. The exported adapter and 768-pixel processor can later be used with a 4-bit base locally; long item lists still need a local GPU memory check.

In [ ]:
# Start with a CUDA-enabled PyTorch installation supplied by the A100 environment.
# Install into this kernel; restart it if Jupyter requests a restart.
%pip install "transformers>=5,<6" "peft>=0.18,<1" "accelerate>=1,<2" "datasets>=4,<6" "huggingface-hub>=1,<3" pillow

In [ ]:
from pathlib import Path
import sys

# Works from the repository root or a notebook subdirectory. Training notebooks
# also remain standalone on an A100 when only the notebook has been uploaded.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "pyproject.toml").is_file()
     and (candidate / "packages/receipt_extraction").is_dir()),
    Path.cwd(),
)
for path in (PROJECT_ROOT, PROJECT_ROOT / "packages"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print("Project directory:", PROJECT_ROOT)


In [ ]:
import copy
import hashlib
import io
import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import torch
from PIL import Image, ImageOps
from datasets import load_dataset
from huggingface_hub import HfApi
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModelForImageTextToText, AutoProcessor, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from safetensors.torch import load_file

MODEL_ID = "HuggingFaceTB/SmolVLM-Base"
DATASET_IDS = ("naver-clova-ix/cord-v2", "naver-clova-ix/cord-v1")  # annotation preference order
# Optionally replace None with a commit SHA to repeat an earlier run exactly.
DATASET_REVISIONS = {name: None for name in DATASET_IDS}
MODEL_REVISION = None
OUTPUT_DIR = (PROJECT_ROOT / "outputs/smolvlm-cord-items-lora")
SEED = 42
IMG_LONGEST_EDGE = 768
BATCH_SIZE, GRAD_ACCUM = 2, 4  # effective batch size 8 on one GPU
EPOCHS, LR = 3, 1e-4
GRADIENT_CHECKPOINTING = True
MAX_STEPS = None              # set 5 for a smoke test; optimizer steps, not receipts
MAX_QUARANTINED_TRAIN_ROWS = 5 # exclude/report only ambiguous training labels; set 0 for strict mode
MAX_SEQUENCE_TOKENS = 8192   # hard check; no silent truncation
MAX_NEW_TOKENS = 2048        # item lists need more than the old four-field 200-token limit
RUN_TEST_EVAL = False         # enable only after choosing the final run using validation
N_TEST = None                # None = full held-out test split
TRAIN_DTYPE = torch.bfloat16

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Use a CUDA GPU with BF16 support, intended: a single A100.")
for name, value in {"BATCH_SIZE": BATCH_SIZE, "GRAD_ACCUM": GRAD_ACCUM, "EPOCHS": EPOCHS,
                    "MAX_SEQUENCE_TOKENS": MAX_SEQUENCE_TOKENS, "MAX_NEW_TOKENS": MAX_NEW_TOKENS}.items():
    if not isinstance(value, int) or value < 1:
        raise ValueError(f"{name} must be a positive integer")
if MAX_STEPS is not None and (not isinstance(MAX_STEPS, int) or MAX_STEPS < 1):
    raise ValueError("MAX_STEPS must be None or a positive integer")
if not isinstance(MAX_QUARANTINED_TRAIN_ROWS, int) or MAX_QUARANTINED_TRAIN_ROWS < 0:
    raise ValueError("MAX_QUARANTINED_TRAIN_ROWS must be a nonnegative integer")
DEVICE = torch.device("cuda:0")
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
props = torch.cuda.get_device_properties(DEVICE)
print(f"GPU: {props.name}, {props.total_memory / 2**30:.1f} GiB")
print(f"Adapter export: {OUTPUT_DIR.resolve()}")

## Target mapping

`menu.nm → description`, `menu.cnt → quantity`, `menu.unitprice → unit_price`, `menu.price → line_total`, and `total.total_price → total`. Preserve printed strings such as `1 x` and `75,000`; do not assume commas are decimal separators or calculate missing prices. Missing scalar fields become `null`.

Preserve nested v2 `sub` entries as `sub_items`. Convert v1 `sub_nm`, `sub_cnt`, `sub_unitprice`, and `sub_price` into the same hierarchy when their associations are unambiguous. Do not promote modifiers such as cooking preferences into independent purchases. The chosen target omits voided items, tax/payment fields, and receipt-level adjustments; item sums therefore need not equal the final total. `line_total` is the printed `price` annotation, not a recomputed post-discount amount.

Conflicting multi-valued scalar labels in training are quarantined with their full annotations and provenance, up to `MAX_QUARANTINED_TRAIN_ROWS` (default 5). No first/last amount is guessed. Set the cap to 0 to require manual correction before continuing. Structural errors, any validation/test conversion errors, and ambiguities beyond the cap stop preparation. Exclusions happen after deduplication so an older version cannot reintroduce the same receipt. The notebook uses the published `gt_parse`, not heuristic reconstruction from bounding boxes.

In [ ]:
ITEM_KEYS = {"description", "quantity", "unit_price", "line_total", "sub_items"}

class AmbiguousAnnotationError(ValueError):
    """Multiple conflicting scalar labels need source-image review, not a guessed value."""
    def __init__(self, field, value):
        self.field = field
        self.value = value
        super().__init__(f"{field}: ambiguous multi-valued or unsupported scalar: {value!r}")

def object_list(value, field):
    if value is None:
        return []
    if isinstance(value, dict):
        return [value]
    if isinstance(value, list) and all(isinstance(item, dict) for item in value):
        return value
    raise ValueError(f"{field}: expected an object or list of objects, got {type(value).__name__}")

def scalar_text(value, field, description=False):
    if value is None or value == "":
        return None
    if isinstance(value, str):
        return value.strip() or None
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, list):
        parts = [scalar_text(part, field) for part in value]
        parts = [part for part in parts if part is not None]
        if not parts:
            return None
        if len(set(parts)) == 1:
            return parts[0]
        if description:
            return " ".join(parts)
        raise AmbiguousAnnotationError(field, value)
    raise ValueError(f"{field}: ambiguous multi-valued or unsupported scalar: {value!r}")

def convert_menu(entry, depth=0):
    if depth > 8:
        raise ValueError("Unexpectedly deep submenu nesting")
    result = {
        "description": scalar_text(entry.get("nm"), "nm", description=True),
        "quantity": scalar_text(entry.get("cnt"), "cnt"),
        "unit_price": scalar_text(entry.get("unitprice"), "unitprice"),
        "line_total": scalar_text(entry.get("price"), "price"),
        "sub_items": [],
    }
    flat_keys = ("sub_nm", "sub_cnt", "sub_unitprice", "sub_price")
    has_flat = any(entry.get(key) not in (None, "", []) for key in flat_keys)
    if entry.get("sub") and has_flat:
        raise ValueError("Both nested and flat submenu annotations exist; inspect before merging")
    if entry.get("sub"):
        result["sub_items"] = [convert_menu(child, depth + 1) for child in object_list(entry["sub"], "sub")]
    elif has_flat:
        # Lists with matching lengths can describe multiple submenus; do not broadcast scalar prices.
        values = {key: entry.get(key) for key in flat_keys}
        lengths = {len(value) for value in values.values() if isinstance(value, list) and value}
        if len(lengths) > 1:
            raise ValueError("Flat submenu lists have inconsistent lengths")
        count = next(iter(lengths), 1)
        for index in range(count):
            child = {}
            for key, value in values.items():
                if isinstance(value, list):
                    selected = value[index] if value else None
                elif count > 1 and value not in (None, ""):
                    raise ValueError("Cannot associate a scalar submenu field with multiple submenus")
                else:
                    selected = value
                child[key.removeprefix("sub_")] = selected
            result["sub_items"].append(convert_menu(child, depth + 1))
    return result

def cord_target(ground_truth):
    gt = json.loads(ground_truth) if isinstance(ground_truth, str) else ground_truth
    parsed = gt.get("gt_parse")
    if not isinstance(parsed, dict):
        raise ValueError("Missing or invalid gt_parse")
    total = parsed.get("total") or {}
    if not isinstance(total, dict):
        raise ValueError("Expected one receipt-level total object")
    return {
        "items": [convert_menu(item) for item in object_list(parsed.get("menu"), "menu")],
        "total": scalar_text(total.get("total_price"), "total.total_price"),
    }

def open_rgb(value):
    if isinstance(value, Image.Image):
        return ImageOps.exif_transpose(value).convert("RGB")
    if isinstance(value, dict):
        value = io.BytesIO(value["bytes"]) if value.get("bytes") is not None else value["path"]
    with Image.open(value) as image:
        return ImageOps.exif_transpose(image).convert("RGB")

def image_fingerprint(image):
    rgb = open_rgb(image)
    digest = hashlib.sha256()
    digest.update(str(rgb.size).encode())
    digest.update(rgb.tobytes())
    return digest.hexdigest()

def split_priority(name):
    return {"train": 0, "validation": 1, "test": 2}[name]

def select_records(candidates):
    """Deduplicate pixels, keep hold-outs protected, and prefer v2 annotation provenance."""
    groups = defaultdict(list)
    for record in candidates:
        groups[record["image_sha256"]].append(record)
    selected, report = [], []
    for fingerprint, group in sorted(groups.items()):
        chosen = min(group, key=lambda row: (DATASET_IDS.index(row["dataset_id"]), -split_priority(row["split"]), row["index"]))
        chosen = dict(chosen)
        chosen["split"] = max((row["split"] for row in group), key=split_priority)
        chosen["aliases"] = [{key: row[key] for key in ("dataset_id", "source_split", "index")} for row in group]
        selected.append(chosen)
        if len(group) > 1:
            report.append({"image_sha256": fingerprint, "kept_dataset": chosen["dataset_id"],
                           "assigned_split": chosen["split"], "aliases": chosen["aliases"],
                           "raw_parse_disagreement": len({json.dumps(row["ground_truth"]["gt_parse"], sort_keys=True) for row in group}) > 1})
    return selected, report

def prepare_records(records, max_quarantined_train_rows=5):
    """Keep audited ambiguity exclusions separate; hold-outs and structural errors are fatal.

    Run after deduplication so an excluded v2 receipt cannot re-enter through v1.
    Source records and downloaded annotations are never modified.
    """
    accepted, quarantined, errors = [], [], []
    for row in records:
        try:
            target = cord_target(row["ground_truth"])
            accepted.append({**row, "target": target,
                             "answer": json.dumps(target, ensure_ascii=False, separators=(",", ":"))})
        except (ValueError, TypeError, KeyError) as exc:
            can_quarantine = (isinstance(exc, AmbiguousAnnotationError)
                              and row["split"] == "train"
                              and len(quarantined) < max_quarantined_train_rows)
            report = {"dataset_id": row["dataset_id"], "source_split": row["source_split"],
                      "assigned_split": row["split"], "index": row["index"],
                      "image_sha256": row["image_sha256"], "aliases": row.get("aliases", []),
                      "error": str(exc), "action": "quarantined" if can_quarantine else "fatal",
                      "gt_parse": row["ground_truth"].get("gt_parse")}
            if can_quarantine:
                quarantined.append(report)
            errors.append(report)
    return accepted, quarantined, errors


In [ ]:
# Download only these two CORD repositories. Resolve and record immutable revisions.
api = HfApi()
resolved_model_revision = api.model_info(MODEL_ID, revision=MODEL_REVISION).sha
resolved_revisions = {name: api.dataset_info(name, revision=DATASET_REVISIONS[name]).sha for name in DATASET_IDS}
raw_datasets, candidates, source_counts = {}, [], {}
for dataset_id in DATASET_IDS:
    dataset = load_dataset(dataset_id, revision=resolved_revisions[dataset_id])
    raw_datasets[dataset_id] = dataset
    for source_split in dataset:
        split = {"train": "train", "validation": "validation", "dev": "validation", "test": "test"}.get(source_split)
        if split is None:
            raise ValueError(f"Unexpected split: {dataset_id}/{source_split}")
        source_counts[f"{dataset_id}/{source_split}"] = len(dataset[source_split])
        for index, row in enumerate(dataset[source_split]):
            ground_truth = json.loads(row["ground_truth"]) if isinstance(row["ground_truth"], str) else row["ground_truth"]
            if not isinstance(ground_truth.get("gt_parse"), dict):
                raise ValueError(f"Missing gt_parse: {dataset_id}/{source_split}/{index}")
            candidates.append({"dataset_id": dataset_id, "source_split": source_split, "split": split,
                               "index": index, "image_sha256": image_fingerprint(row["image"]),
                               "ground_truth": ground_truth})

records, overlap_report = select_records(candidates)
(OUTPUT_DIR / "overlap_report.json").write_text(json.dumps(overlap_report, ensure_ascii=False, indent=2) + "\n")
print("Downloaded source rows:", json.dumps(source_counts, indent=2))
print(f"{len(candidates)} source rows -> {len(records)} unique RGB images")
print("Kept annotation sources:", dict(Counter(row["dataset_id"] for row in records)))
print("Splits before annotation checks:", dict(Counter(row["split"] for row in records)))
print("Duplicate groups with differing annotations:", sum(row["raw_parse_disagreement"] for row in overlap_report))
print("Overlap report:", OUTPUT_DIR / "overlap_report.json")

records, quarantined_records, errors = prepare_records(records, MAX_QUARANTINED_TRAIN_ROWS)
(OUTPUT_DIR / "conversion_errors.json").write_text(json.dumps(errors, ensure_ascii=False, indent=2) + "\n")
(OUTPUT_DIR / "quarantined_annotations.json").write_text(json.dumps(quarantined_records, ensure_ascii=False, indent=2) + "\n")
if quarantined_records:
    print(f"EXCLUDED {len(quarantined_records)} ambiguous training receipts; their duplicates remain excluded.")
    print("Review full annotations and provenance in", OUTPUT_DIR / "quarantined_annotations.json")
    for row in quarantined_records:
        print(f"  {row['dataset_id']}/{row['source_split']}/{row['index']}: {row['error']}")
fatal_errors = [row for row in errors if row["action"] == "fatal"]
if fatal_errors:
    for row in fatal_errors:
        print(f"FATAL {row['dataset_id']}/{row['source_split']}/{row['index']}: {row['error']}")
    raise RuntimeError(
        f"{len(fatal_errors)} fatal conversion errors. Inspect conversion_errors.json. "
        "Hold-out errors, unexpected structures, and exclusions beyond MAX_QUARANTINED_TRAIN_ROWS stop training."
    )

split_rows = {split: [row for row in records if row["split"] == split] for split in ("train", "validation", "test")}
for split, rows in split_rows.items():
    if not rows:
        raise RuntimeError(f"Empty {split} split after deduplication")
    print(split, "receipts:", len(rows), "empty item lists:", sum(not row["target"]["items"] for row in rows),
          "unlabeled totals:", sum(row["target"]["total"] is None for row in rows))
hash_sets = {name: {row["image_sha256"] for row in rows} for name, rows in split_rows.items()}
assert not (hash_sets["train"] & hash_sets["validation"] or hash_sets["train"] & hash_sets["test"] or hash_sets["validation"] & hash_sets["test"])
print(json.dumps(split_rows["train"][0]["target"], ensure_ascii=False, indent=2))

In [ ]:
QUESTION = (
    'Extract all purchased items and the final receipt total as JSON with keys "items" and "total". '
    'Each item must have "description", "quantity", "unit_price", "line_total", and "sub_items". '
    'Keep modifiers and subitems nested in "sub_items" using the same item fields; use [] if none. '
    'Copy the printed strings, including quantity notation and money separators. Use null for missing values. '
    'line_total means the printed item amount, not a recalculated amount. '
    'Exclude voided purchases, merchant details, tax/summary/payment lines and change from items. '
    'Do not infer missing prices or quantities. Return exactly one JSON object.'
)
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=resolved_model_revision, do_image_splitting=True)
processor.image_processor.size = {"longest_edge": IMG_LONGEST_EDGE}
processor.tokenizer.padding_side = "right"
base_config = AutoConfig.from_pretrained(MODEL_ID, revision=resolved_model_revision)
model_context = base_config.text_config.max_position_embeddings
if MAX_SEQUENCE_TOKENS > model_context:
    raise ValueError(f"MAX_SEQUENCE_TOKENS exceeds the model context ({model_context})")
END_TOKEN = "<end_of_utterance>"
end_token_id = processor.tokenizer.convert_tokens_to_ids(END_TOKEN)
if END_TOKEN not in processor.tokenizer.get_vocab():
    raise ValueError("Expected answer terminator missing from tokenizer")
PROMPT = processor.apply_chat_template(
    [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": QUESTION}]}],
    add_generation_prompt=True,
)

class ReceiptDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        record = self.rows[index]
        image = raw_datasets[record["dataset_id"]][record["source_split"]][record["index"]]["image"]
        return {"image": open_rgb(image), "answer": record["answer"]}

def collate(examples):
    images = [[example["image"]] for example in examples]
    prompts = processor(text=[PROMPT] * len(examples), images=images, padding=True, return_tensors="pt")
    full = processor(text=[PROMPT + example["answer"] + END_TOKEN for example in examples],
                     images=images, padding=True, return_tensors="pt")
    lengths = full["attention_mask"].sum(dim=1)
    if int(lengths.max()) > MAX_SEQUENCE_TOKENS:
        raise ValueError("Sequence exceeds the configured limit; no target was truncated")
    labels = full["input_ids"].clone()
    labels[full["attention_mask"] == 0] = -100
    for index in range(len(examples)):
        prompt_length = int(prompts["attention_mask"][index].sum())
        # Fail rather than accidentally train prompt tokens if the tokenizer changes its boundary.
        if not torch.equal(full["input_ids"][index, :prompt_length], prompts["input_ids"][index, :prompt_length]):
            raise RuntimeError("Prompt token boundary differs between prompt-only and answer encoding")
        labels[index, :prompt_length] = -100
    if bool(((labels != -100).sum(dim=1) == 0).any()):
        raise RuntimeError("Example has no supervised answer tokens")
    full["labels"] = labels
    return full

def to_device(batch):
    return {key: value.to(device=DEVICE, dtype=TRAIN_DTYPE if value.is_floating_point() else value.dtype,
                          non_blocking=True) for key, value in batch.items()}

receipt_datasets = {name: ReceiptDataset(rows) for name, rows in split_rows.items()}

In [ ]:
# CPU preflight before loading GPU weights. No examples or partial items are silently truncated.
length_report, oversized = {}, []
for split, dataset in receipt_datasets.items():
    sequence_lengths, answer_lengths = [], []
    for index in range(len(dataset)):
        example = dataset[index]
        encoded = processor(text=PROMPT + example["answer"] + END_TOKEN,
                            images=[[example["image"]]], return_tensors="pt")
        sequence_length = int(encoded["attention_mask"].sum())
        answer_length = len(processor.tokenizer.encode(example["answer"] + END_TOKEN, add_special_tokens=False))
        sequence_lengths.append(sequence_length)
        answer_lengths.append(answer_length)
        split_rows[split][index]["sequence_tokens"] = sequence_length
        split_rows[split][index]["answer_tokens"] = answer_length
        if sequence_length > MAX_SEQUENCE_TOKENS or answer_length > MAX_NEW_TOKENS:
            oversized.append({"split": split, "index": index, "sequence_tokens": sequence_length, "answer_tokens": answer_length})
    length_report[split] = {"max_sequence_tokens": max(sequence_lengths), "max_answer_tokens": max(answer_lengths)}
print(json.dumps(length_report, indent=2))
(OUTPUT_DIR / "length_report.json").write_text(json.dumps({"splits": length_report, "oversized": oversized}, indent=2) + "\n")
if oversized:
    raise RuntimeError(f"{len(oversized)} examples exceed sequence/output limits. Inspect length_report.json and adjust explicit limits; do not truncate JSON.")

# Record exactly which source/version/row and target was used, including collapsed aliases.
manifest = {"datasets": resolved_revisions, "model_id": MODEL_ID, "model_revision": resolved_model_revision,
            "deduplication": "RGB SHA256; v2 annotations preferred; test > validation > train",
            "quarantined_annotations": quarantined_records, "records": []}
for split, rows in split_rows.items():
    for row in rows:
        manifest["records"].append({key: row[key] for key in
                                    ("dataset_id", "source_split", "index", "split", "image_sha256", "aliases", "target", "sequence_tokens", "answer_tokens")})
(OUTPUT_DIR / "dataset_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
b = collate([receipt_datasets["train"][i] for i in range(min(BATCH_SIZE, len(receipt_datasets["train"])))])
print({key: tuple(value.shape) for key, value in b.items()})
print("Supervised answer:", processor.tokenizer.decode(b["labels"][0][b["labels"][0] != -100]))
del b

In [ ]:
# BF16 base + LoRA; no existing SROIE adapter and no k-bit training preparation.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, revision=resolved_model_revision, dtype=TRAIN_DTYPE,
    attn_implementation="sdpa", device_map={"": 0},
)
model.config.use_cache = False
model.config.text_config.use_cache = False
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=r".*text_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.print_trainable_parameters()
print(f"Allocated after load: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

train_loader = DataLoader(receipt_datasets["train"], batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=0, pin_memory=True,
                          generator=torch.Generator().manual_seed(SEED))
validation_loader = DataLoader(receipt_datasets["validation"], batch_size=BATCH_SIZE, shuffle=False,
                               collate_fn=collate, num_workers=0, pin_memory=True)

In [ ]:
import importlib.metadata

training_config = {
    "seed": SEED, "batch_size": BATCH_SIZE, "gradient_accumulation": GRAD_ACCUM,
    "epochs": EPOCHS, "learning_rate": LR, "max_steps": MAX_STEPS,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING, "max_sequence_tokens": MAX_SEQUENCE_TOKENS,
    "model_revision": resolved_model_revision, "datasets": resolved_revisions,
    "max_quarantined_train_rows": MAX_QUARANTINED_TRAIN_ROWS,
    "quarantined_train_receipts": len(quarantined_records),
}

def export_adapter(path, step, validation_loss):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path)
    processor.save_pretrained(path)
    deployment = {
        "model_id": MODEL_ID, "model_revision": resolved_model_revision, "question": QUESTION,
        "image_longest_edge": IMG_LONGEST_EDGE, "max_new_tokens": MAX_NEW_TOKENS,
        "training_precision": "bfloat16", "local_quantization": "nf4",
        "task": "receipt_items_and_total", "schema_version": 1,
        "answer_end_token": END_TOKEN,
    }
    for filename, value in {
        "deployment_config.json": deployment,
        "training_config.json": training_config,
        "training_summary.json": {"optimizer_step": step, "validation_loss": validation_loss,
                                  "smoke_test": MAX_STEPS is not None},
        "package_versions.json": {name: importlib.metadata.version(name) for name in
                                  ("torch", "transformers", "peft", "accelerate", "datasets", "huggingface-hub", "Pillow")},
    }.items():
        (path / filename).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n")

@torch.inference_mode()
def validation_loss():
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for cpu_batch in validation_loader:
        batch = to_device(cpu_batch)
        count = int((batch["labels"][:, 1:] != -100).sum())  # causal loss shifts labels
        with torch.autocast(device_type="cuda", dtype=TRAIN_DTYPE):
            loss = model(**batch).loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite validation loss")
        total_loss += float(loss) * count
        total_tokens += count
        del batch, cpu_batch, loss
    if total_tokens == 0:
        raise RuntimeError("Validation has no supervised tokens")
    return total_loss / total_tokens

In [ ]:
params = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
total_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
if MAX_STEPS is not None:
    total_steps = min(total_steps, MAX_STEPS)
warmup_steps = min(max(1, round(total_steps * 0.05)), max(0, total_steps - 1))
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
optimizer.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
step, best_loss = 0, float("inf")
training_log = []
print(f"Optimizer steps: {total_steps}; effective receipt batch size: {BATCH_SIZE * GRAD_ACCUM}")

for epoch in range(EPOCHS):
    model.train()
    group_loss, group_tokens = 0.0, 0
    for batch_index, cpu_batch in enumerate(train_loader):
        batch = to_device(cpu_batch)
        count = int((batch["labels"][:, 1:] != -100).sum())
        with torch.autocast(device_type="cuda", dtype=TRAIN_DTYPE):
            raw_loss = model(**batch).loss
        if not torch.isfinite(raw_loss):
            raise RuntimeError(f"Non-finite training loss at epoch {epoch + 1}, batch {batch_index}")
        # Accumulate summed answer-token loss; normalize gradients by the actual token count.
        # This handles variable item-list lengths and the final partial accumulation group.
        (raw_loss * count).backward()
        group_loss += raw_loss.detach().float().item() * count
        group_tokens += count
        del batch, cpu_batch, raw_loss
        end_group = (batch_index + 1) % GRAD_ACCUM == 0 or batch_index + 1 == len(train_loader)
        if end_group:
            if group_tokens == 0:
                raise RuntimeError("Accumulation group has no answer tokens")
            for parameter in params:
                if parameter.grad is not None:
                    parameter.grad.div_(group_tokens)
            torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            entry = {"epoch": epoch + 1, "step": step, "train_loss": group_loss / group_tokens}
            training_log.append(entry)
            if step == 1 or step % 10 == 0 or step == total_steps:
                print(f"epoch {epoch + 1} step {step}/{total_steps} loss {entry['train_loss']:.4f} "
                      f"peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
            group_loss, group_tokens = 0.0, 0
            if step >= total_steps:
                break

    # Always preserve the latest completed segment before validation, even for a smoke test.
    export_adapter(OUTPUT_DIR / "last", step, None)
    current_loss = validation_loss()
    training_log.append({"epoch": epoch + 1, "step": step, "validation_loss": current_loss})
    print(f"Validation token loss: {current_loss:.4f}")
    if current_loss < best_loss:
        best_loss = current_loss
        export_adapter(OUTPUT_DIR, step, current_loss)
        print("Saved best validation adapter:", OUTPUT_DIR)
    (OUTPUT_DIR / "training_log.json").write_text(json.dumps(training_log, indent=2) + "\n")
    if step >= total_steps:
        break

del optimizer, scheduler
torch.cuda.empty_cache()
print(f"Done. Best validation loss: {best_loss:.4f}. Export: {OUTPUT_DIR}")
print("A MAX_STEPS-limited export is a smoke/limited run, not evidence of a fully trained model.")

## Inspect the selected adapter

Reload the best validation adapter into the existing model. Generation recognizes the training terminator and stops at a complete JSON object. Validation examples below are diagnostic, not a replacement for the optional held-out evaluation.

The `last/` folder preserves the latest adapter; the root export contains the best validation adapter. These are adapter exports, not optimizer-state checkpoints for exact training resumption. The dataset manifest, overlap report, length report, configuration, package versions, and training log are saved in the root output folder.

In [ ]:
# Restore the best adapter without allocating another base model.
set_peft_model_state_dict(model, load_file(str(OUTPUT_DIR / "adapter_model.safetensors"), device="cpu"))
model.eval()

from transformers import StoppingCriteria, StoppingCriteriaList

def parse_prediction(text):
    value, _ = json.JSONDecoder().raw_decode(text.lstrip())
    if not isinstance(value, dict) or set(value) != {"items", "total"}:
        raise ValueError("Expected items and total")
    def check_items(items, depth=0):
        if not isinstance(items, list) or depth > 8:
            raise ValueError("Invalid item hierarchy")
        for item in items:
            if not isinstance(item, dict) or set(item) != ITEM_KEYS:
                raise ValueError("Invalid item fields")
            if any(item[key] is not None and not isinstance(item[key], str) for key in ITEM_KEYS - {"sub_items"}):
                raise ValueError("Item values must be strings or null")
            check_items(item["sub_items"], depth + 1)
    check_items(value["items"])
    if value["total"] is not None and not isinstance(value["total"], str):
        raise ValueError("Total must be a string or null")
    return value

class CompleteJSON(StoppingCriteria):
    def __init__(self, prompt_length):
        self.prompt_length = prompt_length
    def __call__(self, input_ids, scores, **kwargs):
        text = processor.tokenizer.decode(input_ids[0, self.prompt_length:], skip_special_tokens=True)
        try:
            value, _ = json.JSONDecoder().raw_decode(text.lstrip())
            return isinstance(value, dict)
        except json.JSONDecodeError:
            return False

@torch.inference_mode()
def predict_image(image):
    model.eval()
    inputs = to_device(processor(text=[PROMPT], images=[[open_rgb(image)]], return_tensors="pt"))
    prompt_length = inputs["input_ids"].shape[1]
    if prompt_length + MAX_NEW_TOKENS > model_context:
        raise ValueError("Prompt plus generation budget exceeds the model context")
    eos_ids = sorted({end_token_id, processor.tokenizer.eos_token_id} - {None})
    with torch.autocast(device_type="cuda", dtype=TRAIN_DTYPE):
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                num_beams=1, use_cache=True, eos_token_id=eos_ids,
                                stopping_criteria=StoppingCriteriaList([CompleteJSON(prompt_length)]))
    text = processor.tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True)
    try:
        return {"prediction": parse_prediction(text), "raw_text": text, "valid_json": True}
    except (ValueError, TypeError, json.JSONDecodeError):
        return {"prediction": None, "raw_text": text, "valid_json": False}

In [ ]:
from IPython.display import display
example = receipt_datasets["validation"][0]
preview = example["image"].copy()
preview.thumbnail((600, 900))
display(preview)
print("Reference:", example["answer"])
print(json.dumps(predict_image(example["image"]), ensure_ascii=False, indent=2))

In [ ]:
# Opt-in only. Never choose a checkpoint or tune prompts using these test results.
if RUN_TEST_EVAL:
    def normalize(value):
        return " ".join(value.casefold().split()) if isinstance(value, str) else value
    def item_counter(items):
        # Main purchased lines only; nested modifier accuracy is reflected by full-receipt exact match.
        return Counter((normalize(item["description"]), normalize(item["quantity"]),
                        normalize(item["unit_price"]), normalize(item["line_total"])) for item in items)
    examples = receipt_datasets["test"]
    n = len(examples) if N_TEST is None else min(N_TEST, len(examples))
    if n < 1:
        raise ValueError("N_TEST must be positive or None")
    valid, exact, total_hits, total_labeled = 0, 0, 0, 0
    matched, predicted_count, reference_count = 0, 0, 0
    results = []
    for index in range(n):
        example = examples[index]
        gold = json.loads(example["answer"])
        output = predict_image(example["image"])
        prediction = output["prediction"]
        valid += output["valid_json"]
        exact += prediction == gold
        if gold["total"] is not None:
            total_labeled += 1
            total_hits += prediction is not None and prediction["total"] == gold["total"]
        references = item_counter(gold["items"])
        predictions = item_counter(prediction["items"]) if prediction is not None else Counter()
        matched += sum((references & predictions).values())
        reference_count += sum(references.values())
        predicted_count += sum(predictions.values())
        results.append({"index": index, "image_sha256": split_rows["test"][index]["image_sha256"],
                        "reference": gold, **output})
    precision = matched / predicted_count if predicted_count else 0.0
    recall = matched / reference_count if reference_count else 0.0
    metrics = {"receipts": n, "valid_json_rate": valid / n, "full_receipt_exact_match": exact / n,
               "printed_total_exact_match": total_hits / total_labeled if total_labeled else None,
               "main_item_precision": precision, "main_item_recall": recall,
               "main_item_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0}
    (OUTPUT_DIR / "test_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
    (OUTPUT_DIR / "test_predictions.json").write_text(json.dumps(results, ensure_ascii=False, indent=2) + "\n")
    print(json.dumps(metrics, indent=2))
else:
    print("Held-out generation evaluation skipped. Enable RUN_TEST_EVAL after selecting the final run.")

## Copy the new adapter home

Copy the full `outputs/smolvlm-cord-items-lora/` export to the local computer. The tokenizer, processor, deployment configuration, and adapter weights must stay together. Keep it separate from the old four-field receipt adapter.

With the existing project inference script and CUDA/bitsandbytes dependencies, start with:

```bash
python -m experiments.smolvlm.inference receipt.jpg --adapter outputs/smolvlm-cord-items-lora --no-confidence
```

Use `--no-confidence` because the current confidence formatter was designed for flat receipt fields and does not score each nested item. The saved prompt changes inference to the new items/total task. A long receipt may need a longer output budget and more KV-cache memory; benchmark on the 4 GB GPU. Reducing `--image-longest-edge` to 384 can reduce memory but can hurt small-text reading. Do not silently accept truncated item lists.

A100 validation here uses BF16 weights. Re-evaluate the exported adapter with the local 4-bit base before relying on financial records. Currency-aware parsing, discounts/reconciliation, user review, expense categories, and software integration remain separate follow-up work.